## Imports

In [46]:
from av2.datasets.motion_forecasting.eval import metrics
from pathlib import Path
import pandas as pd
import numpy as np
from av2.datasets.motion_forecasting import scenario_serialization
from av2.datasets.motion_forecasting.eval import metrics
import tqdm

## Load data

In [47]:
data_root = Path("../data")
backbone = "DeMo"
program_name = "simple-logic"
test_ground_truth_path = data_root / "av2" / f"av2_mf_focal_test_annotations.parquet"
test_prediction_path = data_root / "av2_backbone_outputs" / backbone / f"submission_test.parquet"
train_data_path = data_root / "results" / program_name / "val" / backbone / "scored_predictions.parquet"
test_data_path = data_root / "results" / program_name / "test" / backbone / "scored_predictions.parquet"
features_path = data_root / "results" / program_name / "val" / backbone / "features.parquet"
optimal_weights_path = data_root / "results" / program_name / "val" / backbone / "optimal_weights.parquet"

## Run tests

In [3]:
def get_val_gt_trajectory(scenario_id: str) -> pd.DataFrame:
    scenario_path = data_root / "av2" / "val" / scenario_id / f"scenario_{scenario_id}.parquet"
    scenario = scenario_serialization.load_argoverse_scenario_parquet(scenario_path)
    focal_track = next(t for t in scenario.tracks if t.track_id == scenario.focal_track_id)
    future_states = [s for s in focal_track.object_states if not s.observed]
    gt = np.array([[s.position[0], s.position[1]] for s in future_states])
    return pd.DataFrame({
        "scenario_id": [scenario_id],
        "gt_trajectory_x": [gt[:, 0].tolist()],
        "gt_trajectory_y": [gt[:, 1].tolist()],
    })

In [ ]:

backbone_data = pd.read_parquet(test_prediction_path)
scenario_ids = backbone_data.scenario_id.unique()

dfs = []
for scenario_id in tqdm.tqdm(scenario_ids):
    try:
        df = pd.read_parquet(data_root / "results" / "simple-logic" / "test" / "DeMo" / f"{scenario_id}.parquet")
        df["scenario_id"] = scenario_id
        del df["agent_type"]
        del df["covariance"]
        del df["is_moving"]
        del df["min_dist_to_focal"]
        del df["included_in_features"]
        del df["tracs_backbone_independent_op"]
        del df["tracs_backbone_linear_op"]
        del df["tracs_pointwise_scores"]
        dfs.append(df)
    except:
        print(f"Ignored {scenario_id}")
train_data = pd.concat(dfs, ignore_index=True)
train_data.to_parquet(train_data_path)

 21%|██        | 5192/24984 [01:51<07:29, 44.08it/s] 

Ignored 35135250-1ae9-42eb-a2ee-a473813c93ad


 75%|███████▌  | 18780/24984 [06:06<00:29, 211.57it/s]

Ignored c05171ed-f0e7-430f-8916-dc64885f817b


100%|██████████| 24984/24984 [07:33<00:00, 55.06it/s] 


## Fusion Network Architecture

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F_torch

class FusionNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 2)           # w1, w2 are unconstrained
        )

    @staticmethod
    def extract_features(predicted_tracks):
        relative_tracks = predicted_tracks - predicted_tracks[:, :1, :]  # (6, 60, 2)
        scale = relative_tracks.std() + 1e-6
        return (relative_tracks / scale).flatten().astype(np.float32)  # (720,)

    def forward(self, x):
        return self.net(x)

    def predict_weights(self, predicted_tracks):
        x = torch.tensor(FusionNet.extract_features(predicted_tracks)).unsqueeze(0)
        with torch.no_grad():
            return self.forward(x).squeeze(0).numpy()

## Supervisor

In [35]:
def optimal_weights_grid(log_tracs_scores, log_backbone_scores, predicted_tracks, gt_track, n_grid=25):
    """Vectorized grid search over (w1, w2)."""
    
    w = np.linspace(0, 1, n_grid)
    W1, W2 = np.meshgrid(w, w, indexing="ij")
    W1, W2 = W1.flatten(), W2.flatten()

    log_fused = W1[:, None] * log_tracs_scores[None, :] + W2[:, None] * log_backbone_scores[None, :]
    log_fused -= log_fused.max(axis=1, keepdims=True)
    fused = np.exp(log_fused) / np.exp(log_fused).sum(axis=1, keepdims=True)

    brier_ade = []
    for weights in fused:
        brier_ade.append(metrics.compute_brier_ade(predicted_tracks, gt_track, weights).min())

    raw_backbone_score = metrics.compute_brier_ade(predicted_tracks, gt_track, np.exp(log_backbone_scores)).min()
    raw_tracs_score = metrics.compute_brier_ade(predicted_tracks, gt_track, np.exp(log_tracs_scores)).min()

    idx = np.argmin(brier_ade)
    optimized_score = metrics.compute_brier_ade(predicted_tracks, gt_track, fused[idx]).min()

    return W1[idx], W2[idx], raw_backbone_score, raw_tracs_score, optimized_score

## Training

In [ ]:
train_data = pd.read_parquet(train_data_path)

In [ ]:
all_features = []
all_weights = []

for scenario_id in tqdm.tqdm(train_data.scenario_id.unique()):
    gt_row = get_val_gt_trajectory(scenario_id)
    gt_track = np.stack([
        np.array(gt_row["gt_trajectory_x"]),
        np.array(gt_row["gt_trajectory_y"])
    ], axis=1).astype(np.float64)

    scenario_data = train_data[train_data["scenario_id"] == scenario_id]
    log_tracs_scores = np.log(np.array(scenario_data["tracs_trajectory_score"]))
    log_backbone_scores = np.log(np.array(scenario_data["backbone_trajectory_score"]))
    predicted_tracks = np.moveaxis(
        np.array([[*scenario_data["x_traj"]], [*scenario_data["y_traj"]]]),
        source=[0, 1, 2], destination=[2, 0, 1],
    ).astype(np.float64)

    w1, w2, _, _, _ = optimal_weights_grid(log_tracs_scores, log_backbone_scores, predicted_tracks, gt_track)

    all_features.append(FusionNet.extract_features(predicted_tracks))
    all_weights.append([w1, w2])

F = torch.tensor(np.stack(all_features), dtype=torch.float32)
W = torch.tensor(np.array(all_weights), dtype=torch.float32)
print(f"Dataset: {F.shape} features, {W.shape} labels")

  0%|          | 0/24982 [00:00<?, ?it/s]

100%|██████████| 24982/24982 [09:48<00:00, 42.43it/s]

Dataset: torch.Size([24982, 720]) features, torch.Size([24982, 2]) labels


In [ ]:
torch.save(F, features_path)
torch.save(W, optimal_weights_path)

In [43]:
F = torch.load(features_path)
W = torch.load(optimal_weights_path)

model = FusionNet(F.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

for epoch in range(200):
    for idx in torch.randperm(len(F)).split(256):
        optimizer.zero_grad()
        nn.functional.mse_loss(model(F[idx]), W[idx]).backward()
        optimizer.step()
    scheduler.step()
    if epoch % 20 == 0:
        with torch.no_grad():
            loss = nn.functional.mse_loss(model(F), W).item()
            w_pred = model(F).numpy()
            print(f"Epoch {epoch}: loss={loss:.4f}  mean w1={w_pred[:,0].mean():.2f}  mean w2={w_pred[:,1].mean():.2f}")

Epoch 0: loss=0.1050  mean w1=0.02  mean w2=0.71
Epoch 20: loss=0.1025  mean w1=0.04  mean w2=0.69
Epoch 40: loss=0.1019  mean w1=0.03  mean w2=0.71
Epoch 60: loss=0.1017  mean w1=0.03  mean w2=0.72
Epoch 80: loss=0.1015  mean w1=0.02  mean w2=0.71
Epoch 100: loss=0.1016  mean w1=0.03  mean w2=0.70
Epoch 120: loss=0.1014  mean w1=0.04  mean w2=0.71
Epoch 140: loss=0.1011  mean w1=0.03  mean w2=0.72
Epoch 160: loss=0.1010  mean w1=0.03  mean w2=0.73
Epoch 180: loss=0.1008  mean w1=0.03  mean w2=0.72


In [48]:
test_data = pd.read_parquet(test_data_path)
ground_truth = pd.read_parquet(test_ground_truth_path)
backbone_scores = []
fusion_scores = []

for scenario_id in tqdm.tqdm(test_data["scenario_id"].unique()):
    gt_row = ground_truth[ground_truth["scenario_id"] == scenario_id]
    gt_track = np.stack([
        np.array(gt_row["gt_trajectory_x"]),
        np.array(gt_row["gt_trajectory_y"])
    ], axis=1).astype(np.float64)

    scenario_data = test_data[test_data["scenario_id"] == scenario_id]
    backbone_probs = np.array(scenario_data["backbone_trajectory_score"], dtype=np.float64)
    tracs_scores = np.array(scenario_data["tracs_trajectory_score"], dtype=np.float64)
    predicted_tracks = np.moveaxis(
        np.array([[*scenario_data["x_traj"]], [*scenario_data["y_traj"]]]),
        source=[0, 1, 2], destination=[2, 0, 1],
    ).astype(np.float64)

    # Backbone only
    backbone_probs_norm = backbone_probs / backbone_probs.sum()
    backbone_scores.append(metrics.compute_brier_ade(predicted_tracks, gt_track, backbone_probs_norm).min())

    # FusionNet
    w1, w2 = model.predict_weights(predicted_tracks)
    log_fused = w1 * np.log(tracs_scores + 1e-10) + w2 * np.log(backbone_probs + 1e-10)
    fused_probs = np.exp(log_fused - log_fused.max())
    fused_probs /= fused_probs.sum()
    fusion_scores.append(metrics.compute_brier_ade(predicted_tracks, gt_track, fused_probs).min())

backbone_scores = np.array(backbone_scores)
fusion_scores = np.array(fusion_scores)

print(f"Backbone   Brier-minADE: {backbone_scores.mean():.4f}")
print(f"FusionNet  Brier-minADE: {fusion_scores.mean():.4f}")
print(f"Delta: {fusion_scores.mean() - backbone_scores.mean():.4f}  ({(fusion_scores.mean() / backbone_scores.mean() - 1) * 100:.1f}%)")

# Per-scenario improvement distribution
delta = fusion_scores - backbone_scores
print(f"\nImproved: {(delta < 0).mean()*100:.1f}%  Degraded: {(delta > 0).mean()*100:.1f}%")

FileNotFoundError: [Errno 2] No such file or directory: '../data/results/simple-logic/test/DeMo/scored_predictions.parquet'